# Week 3: Transfer Learning + Hyperparameter Optimization
### Continuation of Week 2 — Custom CNN Baseline

---

**Week 2 Recap:**
- Built a custom CNN (128×128 input) → saved as `best_cnn_model.keras`
- Trained with EarlyStopping + ReduceLROnPlateau
- Established a baseline accuracy on the PlantVillage_split dataset

**Week 3 Goals:**
- Load the Week-2 baseline for comparison
- Apply Transfer Learning (MobileNetV2 pretrained on ImageNet)
- Experiment with learning rates (hyperparameter tuning)
- Fine-tune the top layers of the base model
- Evaluate using Accuracy, Precision, Recall + Confusion Matrix
- Compare Week-2 baseline vs Week-3 transfer learning results

## 1. Imports

In [2]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## 2. Paths & Config
> Same dataset paths as Week 1 & 2. Only `IMG_SIZE` changes (128 → 224) to match MobileNetV2's expected input.

In [3]:
# ── Dataset paths (same as Week 1 & 2) ──────────────────────────────
base_dir = Path(r"C:\Users\user\Documents\Agriculture & Smart Farming\PlantVillage_split")

train_dir = base_dir / "train"
val_dir   = base_dir / "val"
test_dir  = base_dir / "test"

# Week-2 model path (used for baseline comparison)
WEEK2_MODEL_PATH = "best_cnn_model.keras"

# Week-3 best model will be saved here
WEEK3_MODEL_PATH = "week3_best_transfer_model.keras"

# ── Config ──────────────────────────────────────────────────────────
# MobileNetV2 requires 224×224 input (upgraded from Week 2's 128×128)
IMG_SIZE   = 224
BATCH_SIZE = 32   # increased from 16; 224px images fit fine in GPU memory
EPOCHS     = 20

NUM_CLASSES = len(os.listdir(train_dir))

print(f"Train : {train_dir}")
print(f"Val   : {val_dir}")
print(f"Test  : {test_dir}")
print(f"\n📊 Number of classes detected: {NUM_CLASSES}")

Train : C:\Users\user\Documents\Agriculture & Smart Farming\PlantVillage_split\train
Val   : C:\Users\user\Documents\Agriculture & Smart Farming\PlantVillage_split\val
Test  : C:\Users\user\Documents\Agriculture & Smart Farming\PlantVillage_split\test

📊 Number of classes detected: 38


## 3. Data Generators
> Augmentation strategy kept consistent with Week 2, with minor additions (shear, brightness) for richer variety.

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.15,
    shear_range=0.1,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest"
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

NUM_CLASSES = train_generator.num_classes

print(f"\n✅ Generators ready")
print(f"   Train batches : {len(train_generator)}")
print(f"   Val batches   : {len(val_generator)}")
print(f"   Test batches  : {len(test_generator)}")
print(f"   Classes       : {NUM_CLASSES}")

Found 37997 images belonging to 38 classes.
Found 8146 images belonging to 38 classes.
Found 8162 images belonging to 38 classes.

✅ Generators ready
   Train batches : 1188
   Val batches   : 255
   Test batches  : 256
   Classes       : 38


## 4. Load Week-2 Baseline for Comparison
> We evaluate the Week-2 custom CNN (128×128) on the test set first, so we have a number to beat.

In [5]:
print("=" * 55)
print("📦 LOADING WEEK-2 BASELINE MODEL FOR COMPARISON")
print("=" * 55)

# Week-2 used 128×128 — build a separate 128×128 test generator just for this
val_test_datagen_128 = ImageDataGenerator(rescale=1./255)
test_generator_128 = val_test_datagen_128.flow_from_directory(
    test_dir,
    target_size=(128, 128),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

try:
    baseline_model = tf.keras.models.load_model(WEEK2_MODEL_PATH)
    baseline_loss, baseline_acc = baseline_model.evaluate(test_generator_128, verbose=0)
    print(f"\n✅ Week-2 Baseline Test Accuracy : {baseline_acc * 100:.2f}%")
    BASELINE_ACC = baseline_acc
except Exception as e:
    print(f"⚠️  Could not load Week-2 model: {e}")
    print("    (Comparison will be skipped at the end)")
    BASELINE_ACC = None

📦 LOADING WEEK-2 BASELINE MODEL FOR COMPARISON
Found 8162 images belonging to 38 classes.

✅ Week-2 Baseline Test Accuracy : 95.14%


## 5. Build Transfer Learning Model
> MobileNetV2 base (frozen) + custom classification head.

In [6]:
def build_transfer_model(learning_rate=0.0001):
    """
    Transfer Learning model using MobileNetV2 pretrained on ImageNet.
    Base layers are frozen; only the custom classification head is trained
    in Phase 1. Fine-tuning (Phase 2) unlocks the last 30 base layers.
    """
    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base_model.trainable = False   # freeze all pretrained layers

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model, base_model

print("✅ build_transfer_model() defined")

✅ build_transfer_model() defined


## 6. Hyperparameter Experiment — Learning Rate Search
> Test two learning rates over 5 quick epochs each (with callbacks). Pick the better one for full training.

In [7]:
print("=" * 55)
print("🔬 HYPERPARAMETER EXPERIMENT: Learning Rate Search")
print("=" * 55)
print("Testing lr = 0.001  vs  lr = 0.0001 (5 quick epochs each)\n")

best_val_acc = 0
best_lr      = None

for lr in [0.001, 0.0001]:
    print(f"\n--- lr = {lr} ---")

    model_exp, _ = build_transfer_model(learning_rate=lr)

    early_stop_exp = EarlyStopping(
        monitor="val_loss", patience=3,
        restore_best_weights=True, verbose=0
    )
    reduce_lr_exp = ReduceLROnPlateau(
        monitor="val_loss", factor=0.3,
        patience=2, min_lr=1e-7, verbose=0
    )

    history_exp = model_exp.fit(
        train_generator,
        validation_data=val_generator,
        epochs=5,
        callbacks=[early_stop_exp, reduce_lr_exp],
        verbose=1
    )

    final_val_acc = max(history_exp.history["val_accuracy"])
    print(f"  → Best val_accuracy : {final_val_acc:.4f}")

    if final_val_acc > best_val_acc:
        best_val_acc = final_val_acc
        best_lr      = lr

print(f"\n✅ Best learning rate selected : {best_lr}  (val_acc = {best_val_acc:.4f})")

🔬 HYPERPARAMETER EXPERIMENT: Learning Rate Search
Testing lr = 0.001  vs  lr = 0.0001 (5 quick epochs each)


--- lr = 0.001 ---
Epoch 1/5
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 1786s 1s/step - accuracy: 0.8293 - loss: 0.5686 - val_accuracy: 0.9238 - val_loss: 0.2303 - learning_rate: 0.0010
Epoch 2/5
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 1927s 2s/step - accuracy: 0.8927 - loss: 0.3420 - val_accuracy: 0.9282 - val_loss: 0.2155 - learning_rate: 0.0010
Epoch 3/5
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 1944s 2s/step - accuracy: 0.9079 - loss: 0.2903 - val_accuracy: 0.9419 - val_loss: 0.1752 - learning_rate: 0.0010
Epoch 4/5
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 1907s 2s/step - accuracy: 0.9144 - loss: 0.2671 - val_accuracy: 0.9437 - val_loss: 0.1742 - learning_rate: 0.0010
Epoch 5/5
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 1774s 1s/step - accuracy: 0.9175 - loss: 0.2580 - val_accuracy: 0.9386 - val_loss: 0.1973 - learning_rate: 0.0010
  → Best val_accuracy : 0.9437

--- lr = 0.0001 ---
Epoch 1/5
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 198

## 7. Phase 1 — Full Training (Frozen Base)
> Train the classification head for up to 20 epochs using the best learning rate found above.

In [ ]:
print("=" * 55)
print("🚀 PHASE 1 — FULL TRAINING WITH BEST LR (frozen base)")
print("=" * 55)

model, base_model = build_transfer_model(learning_rate=best_lr)
model.summary()

early_stop_p1 = EarlyStopping(
    monitor="val_loss", patience=6,
    restore_best_weights=True, verbose=1
)
checkpoint_p1 = ModelCheckpoint(
    WEEK3_MODEL_PATH, monitor="val_accuracy",
    save_best_only=True, verbose=1
)
reduce_lr_p1 = ReduceLROnPlateau(
    monitor="val_loss", factor=0.3,
    patience=3, min_lr=1e-7, verbose=1
)

history_transfer = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stop_p1, checkpoint_p1, reduce_lr_p1],
    verbose=1
)

print("✅ Phase 1 training complete")

🚀 PHASE 1 — FULL TRAINING WITH BEST LR (frozen base)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 38)             │         9,766 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,600,806 (9.92 MB)

 Trainable params: 340,262 (1.30 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

Epoch 1/20
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7509 - loss: 0.9194
Epoch 1: val_accuracy improved from None to 0.91787, saving model to week3_best_transfer_model.keras
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 1669s 1s/step - accuracy: 0.8293 - loss: 0.5820 - val_accuracy: 0.9179 - val_loss: 0.2524 - learning_rate: 0.0010
Epoch 2/20
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8951 - loss: 0.3346
Epoch 2: val_accuracy improved from 0.91787 to 0.93125, saving model to week3_best_transfer_model.keras
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 1777s 1s/step - accuracy: 0.8934 - loss: 0.3356 - val_accuracy: 0.9313 - val_loss: 0.2076 - learning_rate: 0.0010
Epoch 3/20
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9093 - loss: 0.2915
Epoch 3: val_accuracy improved from 0.93125 to 0.93899, saving model to week3_best_transfer_model.keras
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 1655s 1s/step - accuracy: 0.9077 - loss: 0.2932 - val_accuracy: 0.9390 - val_loss: 0.1835 - learning_rate: 0.

## 8. Phase 2 — Fine-Tuning
> Unfreeze the last 30 MobileNetV2 layers and train with a very small learning rate (5e-6) to avoid destroying pretrained weights.
>
> ⚠️ Fresh callbacks are created — never reuse Phase 1 callbacks as they carry stale internal state.

In [ ]:
print("=" * 55)
print("🔧 PHASE 2 — FINE-TUNING (last 30 MobileNetV2 layers)")
print("=" * 55)

base_model.trainable = True

# Keep earlier feature-extractor layers frozen
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with a very small lr to avoid destroying pretrained weights
model.compile(
    optimizer=Adam(learning_rate=5e-6),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Fresh callbacks — never reuse Phase 1 callbacks
early_stop_p2 = EarlyStopping(
    monitor="val_loss", patience=6,
    restore_best_weights=True, verbose=1
)
checkpoint_p2 = ModelCheckpoint(
    WEEK3_MODEL_PATH, monitor="val_accuracy",
    save_best_only=True, verbose=1
)
reduce_lr_p2 = ReduceLROnPlateau(
    monitor="val_loss", factor=0.3,
    patience=3, min_lr=1e-8, verbose=1
)

history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop_p2, checkpoint_p2, reduce_lr_p2],
    verbose=1
)

print("✅ Phase 2 fine-tuning complete")

## 9. Load Best Model & Evaluate

In [ ]:
print("=" * 55)
print("📊 EVALUATING BEST WEEK-3 MODEL")
print("=" * 55)

best_model = tf.keras.models.load_model(WEEK3_MODEL_PATH)

val_loss, val_acc   = best_model.evaluate(val_generator,  verbose=0)
test_loss, test_acc = best_model.evaluate(test_generator, verbose=0)

print(f"\nValidation Accuracy : {val_acc  * 100:.2f}%")
print(f"Test Accuracy       : {test_acc * 100:.2f}%")

## 10. Classification Report — Precision, Recall & F1

In [ ]:
print("=" * 55)
print("📋 CLASSIFICATION REPORT")
print("=" * 55)

test_generator.reset()
y_pred_probs = best_model.predict(test_generator, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = test_generator.classes
class_names  = list(test_generator.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))

## 11. Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(16, 14))
    sns.heatmap(
        cm,
        annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title("Confusion Matrix – Test Set (Week 3 Transfer Learning)", fontsize=14)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig("week3_confusion_matrix.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("✅ Confusion matrix saved → week3_confusion_matrix.png")

plot_confusion_matrix(y_true, y_pred, class_names)

## 12. Training Curves — Phase 1 & Phase 2

In [ ]:
def plot_history(history, title):
    acc      = history.history["accuracy"]
    val_acc  = history.history["val_accuracy"]
    loss     = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc,     label="Train Accuracy")
    plt.plot(epochs_range, val_acc, label="Val Accuracy")
    plt.title(title + " – Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss,     label="Train Loss")
    plt.plot(epochs_range, val_loss, label="Val Loss")
    plt.title(title + " – Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    fname = title.replace(" ", "_").replace("(", "").replace(")", "") + ".png"
    plt.savefig(fname, dpi=100, bbox_inches="tight")
    plt.show()
    print(f"✅ Training curve saved → {fname}")

plot_history(history_transfer, "Phase 1 Transfer Learning")
plot_history(history_finetune,  "Phase 2 Fine Tuning")

## 13. Week-by-Week Progress Summary

In [ ]:
print("=" * 55)
print("📈 WEEK-BY-WEEK PROGRESS SUMMARY")
print("=" * 55)

if BASELINE_ACC is not None:
    improvement = (test_acc - BASELINE_ACC) * 100
    print(f"  Week 2  – Custom CNN (128×128)           : {BASELINE_ACC * 100:.2f}%")
    print(f"  Week 3  – MobileNetV2 Transfer (224×224) : {test_acc * 100:.2f}%")
    print(f"  Improvement                              : +{improvement:.2f}%")
else:
    print(f"  Week 3 – MobileNetV2 Transfer Learning Test Accuracy: {test_acc * 100:.2f}%")
    print("  (Week-2 baseline not available for comparison)")

print("\n✅ Week 3 complete!")
print(f"   Best model saved at : {WEEK3_MODEL_PATH}")